# Chapter 1 — SDK Schema, Check, and Diagnose

This notebook walks through the foundation of the FactPy round story:

1. **SDK schema authoring** — define a `Person` entity using `kernel.sdk` declarative DSL.
2. **Q1 Check** (`check_derivation_binding`) — ask whether a specific binding satisfies a derivation.
3. **Q2 Diagnose** (`diagnose_derivation_binding`) — when Check fails, locate the responsible body atom.

All code is imported from the assertion-bearing script `examples/round_story_full_demo.py`. The notebook does not redefine schemas or fixtures — it walks through the same scenario that the smoke test exercises.

**Series navigation**

- This is chapter 1 of 4. Next: `02_overlay_why_not_frontier.ipynb`.
- The full integrated walkthrough is `round_story_full_demo.py` (run as a script for end-to-end smoke verification).

## Setup

Make `src/` and `examples/` importable from any launch directory, then load the demo module.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'examples').exists() and (_repo_root.parent / 'examples').exists():
    _repo_root = _repo_root.parent
for sub in ('src', 'examples'):
    candidate = _repo_root / sub
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import round_story_full_demo as demo  # noqa: E402

## 1. SDK schema fixture

The demo defines a `Person` entity with three fields (`name` as identity, `age`, `region`) and seeds a small in-memory ledger with four people (alice / bob / carol / dave). `_build_fixture()` returns a frozen `DemoFixture` carrying the live `Store`, the schema index, the compiled derivation plan, and the seeded people.

Reading the fixture is enough to learn the data shape the rest of the chapter operates on.

In [ ]:
fixture = demo._build_fixture()

print(f'Seeded {len(fixture.people)} people: {sorted(fixture.people)}')
print(f'Derivation target: {fixture.plan.heads[0].target_pred_id}')
print(f'Head variables: {fixture.plan.heads[0].head_var_names}')
alice = fixture.people["alice"]
print(f'\nAlice: e_ref={alice.e_ref!r}, age={alice.age}, region={alice.region!r}')

## 2. Q1 Check — does this binding pass?

`check_derivation_binding(...)` evaluates a derivation plan against an explicit binding and returns one of `passed | failed | unsupported | invalid_request`. A successful Check carries:

- `matched_count` — how many native-evaluator rows satisfied the binding
- `matched_binding` — the canonical binding accepted by the plan
- `evidence_envelope` — the per-engine support carrier (a `SupportArtifact` for the native engine)

The phase below asks whether Alice (age=25, region='us') passes the eligibility derivation.

In [ ]:
check_request, check_result, support_artifact = demo._phase_check(fixture, verbose=True)

print(f'\nstatus           = {check_result.status}')
print(f'matched_count    = {check_result.matched_count}')
print(f'matched_binding  = {check_result.matched_binding}')
print(f'support kind     = {type(support_artifact).__name__}')

## 3. Q2 Diagnose — when Check fails, where does it fail?

`diagnose_derivation_binding(...)` inspects a failing binding and returns a `DiagnoseAtomLocator` pointing at the first body atom that does not match. The locator carries `branch_index`, `failed_atom_index`, and the `attempted_binding` at the failure frontier.

Below, Alice's age is forced to `99` (the rule requires age == 25), so Diagnose pinpoints the age atom.

In [ ]:
diagnose_request, diagnose_result = demo._phase_diagnose(fixture, verbose=True)
locator = diagnose_result.diagnostic_payload

print(f'\nstatus               = {diagnose_result.status}')
print(f'failure_kind         = {diagnose_result.failure_kind}')
print(f'branch_index         = {locator.branch_index}')
print(f'failed_atom_index    = {locator.failed_atom_index}')
print(f'attempted_binding    = {locator.attempted_binding}')

## 4. Aggregate verification

The chapter helper `run_sdk_check_diagnose_demo(...)` re-runs the same two phases against a fresh fixture and returns a stable status dict. The unittest `test_examples_round_story_full_demo` asserts on this dict, so any behavioral change in the SDK / Check / Diagnose surface fails fast.

In [ ]:
summary = demo.run_sdk_check_diagnose_demo(verbose=False)
expected = {'check': 'passed', 'diagnose': 'atom_localized'}

print(f'Chapter summary : {summary}')
print(f'Expected        : {expected}')
assert summary == expected, summary
print('\n✓ Chapter 1 aggregate matches expected smoke contract.')

## Where to next

- **Chapter 2 (`02_overlay_why_not_frontier.ipynb`)** — what-if fact overlays (Q3), finite-set Why-not diagnosis (Q4), and the evaluator-layer Frontier trace (Q5).
- **Module reference** — `src/kernel/application/docs/01_overview.md` (Check / Diagnose / 8-capability surface).
- **Public boundary** — Check and Diagnose are advanced-importable (`kernel.application.derivation_check_runtime` / `diagnose_runtime`); v0.1 does not ship dedicated SDK shells per the Batch 8 public-surface decision.